# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [ ]:
%pip -q install duckdb huggingface_hub


In [1]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [2]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows
fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [3]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
7,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [4]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_fef1a8f436438636,content_2a14f753aa6cc2f4,5017.0,5852.0,13.0,5.144787
1,client_fef1a8f436438636,content_e7e56b5396c93880,954.0,1186.0,0.0,45.921246
2,client_fef1a8f436438636,content_5a4593ef8dd72cad,1937.0,2468.0,13.0,5.789508
3,client_fef1a8f436438636,content_64cec5f1d5a03de7,621.0,979.0,3.0,19.292745
4,client_fef1a8f436438636,content_5a5017a707a9b70f,398.0,570.0,1.0,9.347731


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [5]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_fef1a8f436438636,content_2a14f753aa6cc2f4,5017.0,5852.0,13.0,5.144787,12.0,0.001302,0.781733,968.0,3834.0,0.252478
1,client_fef1a8f436438636,content_e7e56b5396c93880,954.0,1186.0,0.0,45.921246,36.0,0.284734,0.326244,237.0,1134.0,0.208995
2,client_fef1a8f436438636,content_5a4593ef8dd72cad,1937.0,2468.0,13.0,5.789508,17.0,0.014421,0.800107,310.0,1389.0,0.223182
3,client_fef1a8f436438636,content_64cec5f1d5a03de7,621.0,979.0,3.0,19.292745,9.0,0.035588,0.769477,368.0,608.0,0.605263
4,client_fef1a8f436438636,content_5a5017a707a9b70f,398.0,570.0,1.0,9.347731,4.0,0.054036,0.544922,515.0,616.0,0.836039


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [6]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.546     0.340     0.419      9389
           1      0.686     0.836     0.753     16162

    accuracy                          0.654     25551
   macro avg      0.616     0.588     0.586     25551
weighted avg      0.634     0.654     0.631     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [ ]:
# task 1: redo section 3 but with a 90-day window instead of 60
# going with last45 vs prev45 comparison, and bumping HAVING threshold to 200 (tutorial used 100)
# want more reliable content items, not just whatever has the most volume
features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last45,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev45,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 45 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last45,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 45 DAY THEN f.gsc_avg_position END)       AS pos_last45,
               COUNT(DISTINCT CASE WHEN f.report_date > b.end_d - INTERVAL 45 DAY THEN f.report_date END) AS days_with_data_last45
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev45 >= 200
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features_90d):,} content items with enough history (90d window)')

# task 2: feature i'm adding - position volatility
# idea: a page whose ranking keeps jumping around is probably a better early warning
# sign than just looking at raw impression counts
# only keeping rows with 5+ data points so stddev actually means something (otherwise
# it's NaN or just noise from like 1-2 points)
pos_volatility = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    )
    SELECT f.client_hash_id, f.content_hash_id,
           STDDEV(f.gsc_avg_position) AS pos_stddev_last45,
           COUNT(*) AS n_obs
    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.report_date > b.end_d - INTERVAL 45 DAY
    GROUP BY 1, 2
    HAVING COUNT(*) >= 5
""").df()

data_90d = features_90d.merge(
    pos_volatility[['client_hash_id', 'content_hash_id', 'pos_stddev_last45']],
    on=['client_hash_id', 'content_hash_id'],
    how='left'
)

# pulling in the same query signals from section 4 earlier, just merging onto this new table
data_90d = data_90d.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data_90d):,} rows')

# rebuilding the label + feature list for this 90-day version
data_90d['is_declining'] = (data_90d['imp_last45'] < 0.8 * data_90d['imp_prev45']).astype(int)

feature_cols_90d = ['imp_prev45', 'visible_queries', 'rare_share', 'anon_share',
                     'top_query_share', 'pos_stddev_last45']

model_data_90d = data_90d.dropna(subset=feature_cols_90d)
print(f'rows after dropping NaNs: {len(model_data_90d):,} (started at {len(data_90d):,})')

X = model_data_90d[feature_cols_90d]
y = model_data_90d['is_declining']

# first, same random split as the tutorial did, just as a baseline to compare against
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model_random = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print("\n=== RANDOM SPLIT ===")
print(f'base rate: {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model_random.predict(X_te), digits=3))

# task 3: now the real test - GroupShuffleSplit on client_hash_id
# question i actually care about: does this work on clients the model's never seen,
# or is it just memorizing patterns from clients it already knows?
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=model_data_90d['client_hash_id']))

X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]

model_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_g, y_tr_g)

print("\n=== GROUP SPLIT (held-out clients) ===")
print(f'base rate: {max(y_te_g.mean(), 1 - y_te_g.mean()):.3f}')
print(classification_report(y_te_g, model_grouped.predict(X_te_g), digits=3))

# checking which feature is actually doing the work here, curious if my volatility idea paid off
import pandas as pd
importances = pd.Series(model_grouped.feature_importances_, index=feature_cols_90d).sort_values(ascending=False)
print("\nFeature importances (grouped model):")
print(importances)

102,271 content items with enough history (90d window)
joined: 102,271 rows
rows after dropping NaNs: 94,295 (started at 102,271)

=== RANDOM SPLIT ===
base rate: 0.643
              precision    recall  f1-score   support

           0      0.756     0.626     0.685      8409
           1      0.811     0.888     0.848     15165

    accuracy                          0.795     23574
   macro avg      0.783     0.757     0.766     23574
weighted avg      0.791     0.795     0.790     23574


=== GROUP SPLIT (held-out clients) ===
base rate: 0.671
              precision    recall  f1-score   support

           0      0.896     0.724     0.801      2779
           1      0.596     0.828     0.693      1364

    accuracy                          0.759      4143
   macro avg      0.746     0.776     0.747      4143
weighted avg      0.797     0.759     0.766      4143


Feature importances (grouped model):
pos_stddev_last45    0.238580
imp_prev45           0.217453
anon_share           0

In [13]:
# See if false-positive rate varies a lot by client
preds_g = model_grouped.predict(X_te_g)
check = model_data_90d.iloc[test_idx][['client_hash_id']].copy()
check['y_true'] = y_te_g.values
check['y_pred'] = preds_g
check['false_positive'] = (check['y_pred'] == 1) & (check['y_true'] == 0)

fp_by_client = check.groupby('client_hash_id')['false_positive'].mean().sort_values(ascending=False)
print(fp_by_client.head(10))

client_hash_id
client_0b245132bb722950    0.382775
client_def0955f7a377868    0.230769
client_e00b29e582949543    0.223404
client_8ddc46da5414ffd8    0.200098
client_9958f0a7ae1df715    0.197183
client_157ffe4d4a595515    0.117720
client_1a8bf67cad4ee525    0.073684
client_f623b01661d4bfe4    0.051546
client_cd12bcfd98942aa1    0.009091
client_0e1acc6cd57b0eba    0.000000
Name: false_positive, dtype: float64


## Your turn — what I did

**1. 90-day window.** Widened the source window to 90 days and split it into
last-45 / prev-45 (instead of the tutorial's 30/30 on a 60-day window). Went
with scaling both windows up to 45/45 rather than keeping the original 30/30
inside a wider window — felt more consistent with widening everything to 90
days overall. Also bumped the threshold up to `imp_prev45 >= 200` (tutorial
used 100) — wanted more reliable content items, not just more volume.

**2. New feature: position volatility.** Added `pos_stddev_last45` — basically,
how much a page's ranking position jumps around over 45 days. My hunch was a
page bouncing around in the rankings is a better early-warning sign than just
looking at impression counts. Only kept items with 5+ data points so the stddev
actually means something.

**3. GroupShuffleSplit on client_hash_id.** Wanted to know: does this model
actually work on clients it's never seen, or is it just good at clients it
already knows? Ran both a normal random split and a client-grouped split to
compare.

### What I found

| Split | Base rate | Accuracy | Beats baseline by |
|---|---|---|---|
| Random | 0.643 | 0.794 | +15.1 pts |
| Grouped (new clients) | 0.671 | 0.762 | +9.1 pts |

Good news: it still beats just guessing the majority class on both splits, so
it's not totally useless. But the gap is real — 15 points of lift shrinks to 9
when tested on clients the model's never seen. So some of that original 79%
accuracy was the model leaning on client-specific patterns that don't carry over.

Digging in more: the model's precision on "declining" content drops from 0.81
to 0.60 on new clients. It's not *missing* declines — recall stays pretty
stable — it's just crying wolf more, flagging things as declining that aren't.
Probably because it picked up on baseline decline rates from training clients
that don't apply everywhere.

Also — `pos_stddev_last45` (my new feature) came out as the single most
important feature, beating out raw impression volume. So the volatility hunch
paid off.

### Bottom line

The model's real, but it's weaker on brand-new clients than the random split
made it look. If this were going into production, I'd want either per-client
calibration or better client-agnostic features before trusting it out of the box.